In [2]:
import concurrent.futures
import dataclasses
import io

import fsspec
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
def read_speedhive_csv(url: str) -> pd.DataFrame:
    with fsspec.open(url) as f:
        data = f.read()
        data = data.replace(b'",', b" ")
    return pd.read_csv(io.BytesIO(data))


def parse_laptime_seconds(s: pd.Series) -> pd.Series:
    """Parse '1:26.354' / '58.201' style strings to seconds (float)."""
    parts = s.str.rsplit(":", n=3, expand=True)
    isna = parts[1].isna()
    parts[1] = parts[1].where(~isna, parts[0])
    parts[0] = parts[0].where(~isna, 0)
    return pd.to_timedelta(
        parts[0].astype(float) * 60 + parts[1].astype(float), unit="s"
    )


@dataclasses.dataclass
class SpeedhiveSession:
    session_id: int
    name: str

    def base_url(self) -> str:
        return f"https://eventresults-api.speedhive.com/api/v0.2.3/eventresults/sessions/{self.session_id}"

    def results_df(self) -> pd.DataFrame:
        return (
            read_speedhive_csv(f"{self.base_url()}/csv")
            .set_index("Pos")
            .rename_axis(index=None)
        )

    def competitor_data(self, pos: int) -> pd.DataFrame:
        with fsspec.open(f"{self.base_url()}/lapdata/{pos}/csv", "rb") as f:
            data = f.read()

        df = pd.read_csv(io.BytesIO(data.replace(b'",', b" ")), header=0)
        df.iloc[16:20]

        df = df.assign(lap_time=lambda d: d["Lap Time"].pipe(parse_laptime_seconds))
        return df

    def laptimes(self) -> pd.DataFrame:
        results_df = self.results_df()
        with concurrent.futures.ThreadPoolExecutor(max_workers=8) as executor:
            return pd.DataFrame(
                dict(
                    executor.map(
                        lambda i_row: (
                            i_row[1]["Competitor"],
                            self.competitor_data(i_row[0])["lap_time"],
                        ),
                        results_df.iterrows(),
                    )
                )
            )


mk_6hr = SpeedhiveSession(session_id=11880076, name="2026-jul-sp-3h")
laptimes = mk_6hr.laptimes()
mk_6hr.results_df()

,Start Number,Competitor,Class,Total Time,Diff,Laps,Best Lap,Best Lap No.,Best Speed
1,118,Clean Bulls Racing,2026 Endurance 2 Stroke,6:0:46.399,0.000,297,1:6.608,11,73.505 km/h
2,111,James King,2026 Endurance 2 Stroke,6:0:52.018,5.619,297,1:6.700,17,73.403 km/h
3,116,Triple Threat,2026 Endurance 2 Stroke,6:1:11.654,1 lap,296,1:7.167,9,72.893 km/h
4,105,ASH.KING,2026 Endurance 2 Stroke,6:1:37.911,3 laps,294,1:7.397,6,72.644 km/h
5,123,Phoenix DMAX,2026 Endurance 2 Stroke,6:1:38.379,5 laps,292,1:7.035,75,73.036 km/h
6,104,Apex Nova Racing,2026 Endurance 2 Stroke,6:1:51.629,6 laps,291,1:7.289,6,72.761 km/h
7,121,GKA Racing,2026 Endurance 2 Stroke,6:1:3.611,10 laps,287,1:7.526,32,72.505 km/h
8,113,Nomex Racing,2026 Endurance 2 Stroke,6:1:46.037,11 laps,286,1:7.373,19,72.670 km/h
9,117,The Mandem,2026 Endurance 2 Stroke,6:1:55.169,12 laps,285,1:7.760,204,72.255 km/h
10,106,5 Dollar Head,2026 Endurance 2 Stroke,6:1:51.056,17 laps,280,1:7.519,20,72.513 km/h


In [41]:
def plot_gap_to_mean_leader(laptimes: pd.DataFrame) -> go.Figure:
    leader_mean_lap = laptimes.iloc[:, 0].mean()
    return px.line(
        laptimes.assign(
            **{
                col: lambda d, col=col: (
                    (leader_mean_lap - d[col]).cumsum().dt.total_seconds()
                )
                for col in laptimes.columns
            }
        ),
        title="Gap to leader mean lap time (cumulative) - 2026 Apr Dayona MK 6h",
    )


plot_gap_to_mean_leader(laptimes)

In [25]:
king = mk_6hr.competitor_data(2)
triple = mk_6hr.competitor_data(3)

In [33]:
king

,Lap,Pos,Lap Time,Diff to Last Lap,Diff to Best Lap,Gap in Front,Diff to P1,Speed,lap_time
0,1,3,1:10.929,0.000,4.229,0.274,1.041,69.027 km/h,0 days 00:01:10.929000
1,2,3,1:8.117,-,1.417,0.319,1.271,71.876 km/h,0 days 00:01:08.117000
2,3,3,1:11.418,3.301,4.718,0.328,1.404,68.554 km/h,0 days 00:01:11.418000
3,4,3,1:7.175,-,0.475,0.326,1.857,72.884 km/h,0 days 00:01:07.175000
4,5,3,1:7.953,0.778,1.253,0.949,2.598,72.050 km/h,0 days 00:01:07.953000
...,...,...,...,...,...,...,...,...,...
290,291,2,1:9.071,0.084,2.371,1 lap,1 lap,70.884 km/h,0 days 00:01:09.071000
291,292,2,1:8.743,-,2.043,1 lap,1 lap,71.222 km/h,0 days 00:01:08.743000
292,293,2,1:8.216,-,1.516,1 lap,1 lap,71.772 km/h,0 days 00:01:08.216000
293,294,2,1:7.888,-,1.188,1 lap,1 lap,72.119 km/h,0 days 00:01:07.888000


In [40]:
px.line(
    triple["Gap in Front"]
    .where(lambda s: ~s.str.contains(" "))
    .pipe(parse_laptime_seconds)
    .dt.total_seconds(),
    line_shape="hv",
)

In [39]:
def plot_laptime_distribution(laptimes: pd.DataFrame, limit: int | None) -> go.Figure:
    best_lap = laptimes.where(lambda d: d > d.median().min() * 0.93).min().min()
    return (
        laptimes.where(lambda d: (d > best_lap * 0.93) & (d < best_lap * 1.07))
        .assign(
            **{
                col: lambda d, c=col: d[c].dt.total_seconds()
                for col in laptimes.columns
            }
        )[laptimes.columns[:limit]]
        .pipe(px.violin)
    )


plot_laptime_distribution(laptimes, 3)